# Parte 5 — Questões de negócio: Nível 3 (cruzamentos)

**Q8 a Q10.** `tipo_pista` × clima, as BRs mais letais à noite e os períodos
de feriado.
Mesmas regras: taxa de letalidade = % de acidentes fatais do grupo, comparada à
taxa global de **7,16%**, com volume mínimo declarado e leitura sem causalidade.


## 0. Base analítica da Parte 4


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
ARQ_ANALITICA = DIR_TRATADOS / "base_analitica_prf_2023_2025.csv"
df = pd.read_csv(ARQ_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA,
                 dtype={"id": "string"}, low_memory=False)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

TAXA_GLOBAL = df["acidente_fatal"].mean()
print(f"Base: {df.shape[0]:,} acidentes | taxa global de fatais: {TAXA_GLOBAL:.4%}")


Base: 213,497 acidentes | taxa global de fatais: 7.1631%


In [3]:
# --- funções de apoio usadas nas questões ---
def indicadores(base: pd.DataFrame, coluna: str, minimo: int = 0) -> pd.DataFrame:
    """Acidentes, fatais, mortos e taxa de letalidade por categoria.

    Neste exercício, taxa de letalidade = % de acidentes fatais do grupo
    (definição do enunciado), comparada com a taxa global.
    """
    tab = base.groupby(coluna).agg(
        acidentes=("acidente_fatal", "size"),
        fatais=("acidente_fatal", "sum"),
        mortos=("mortos", "sum"),
    )
    tab = tab[tab["acidentes"] >= minimo]
    tab["taxa_letalidade"] = tab["fatais"] / tab["acidentes"]
    tab["lift"] = tab["taxa_letalidade"] / TAXA_GLOBAL
    return tab.sort_values("taxa_letalidade", ascending=False)


def fmt(tab: pd.DataFrame) -> pd.DataFrame:
    """Formata percentuais para leitura (ignora colunas ausentes)."""
    saida = tab.copy()
    if "taxa_letalidade" in saida.columns:
        saida["taxa_letalidade"] = saida["taxa_letalidade"].map("{:.2%}".format)
    if "lift" in saida.columns:
        saida["lift"] = saida["lift"].map("{:.2f}".format)
    return saida


---
## Q8 — `tipo_pista` × condição meteorológica:
como a letalidade se combina entre pista e clima? Onde está a pior combinação?


In [4]:
# Ignorado fica fora da comparação de clima: não é uma condição meteorológica.
clima = df[df["condicao_metereologica"] != "Ignorado"]

contagem = clima.pivot_table(
    index="tipo_pista", columns="condicao_metereologica",
    values="acidente_fatal", aggfunc="size", fill_value=0,
)
taxa = clima.pivot_table(
    index="tipo_pista", columns="condicao_metereologica",
    values="acidente_fatal", aggfunc="mean",
)
valida = contagem >= 50          # volume mínimo por célula

print("Acidentes por combinação (célula sem volume suficiente aparece como —)")
print(contagem.where(valida).to_string())
print(f"\nCélulas com pelo menos 50 acidentes: {int(valida.values.sum())} de {contagem.size}")


Acidentes por combinação (célula sem volume suficiente aparece como —)
condicao_metereologica  Chuva  Céu Claro  Garoa/Chuvisco  Granizo  Neve  Nevoeiro/Neblina  Nublado   Sol  Vento
tipo_pista                                                                                                     
Dupla                    9531      56082            3386      NaN   NaN               627    13607  5150    152
Múltipla                 1626      12696             756      NaN   NaN                92     3739  1358     61
Simples                  9405      67036            3402      NaN   NaN              1075    15254  5551    115

Células com pelo menos 50 acidentes: 21 de 27


In [5]:
print("Taxa de letalidade por combinação (%)")
display((taxa.where(valida) * 100).round(2))

print("Margens (todas as condições, volume >= 50):")
for coluna in ("tipo_pista", "condicao_metereologica"):
    margem = indicadores(clima, coluna, minimo=50)
    margem = margem[margem.index != "Neve"]
    print(f"\n-- {coluna} --")
    display(fmt(margem)[["acidentes", "fatais", "taxa_letalidade", "lift"]])


Taxa de letalidade por combinação (%)


condicao_metereologica,Chuva,Céu Claro,Garoa/Chuvisco,Granizo,Neve,Nevoeiro/Neblina,Nublado,Sol,Vento
tipo_pista,,,,,,,,,
Dupla,3.22,4.98,3.75,NaN,NaN,8.77,5.00,3.55,3.95
Múltipla,3.51,4.13,3.84,NaN,NaN,7.61,4.63,3.46,4.92
Simples,9.60,10.15,7.50,NaN,NaN,13.95,9.28,8.34,13.91


Margens (todas as condições, volume >= 50):

-- tipo_pista --


,acidentes,fatais,taxa_letalidade,lift
tipo_pista,,,,
Simples,101843,10009,9.83%,1.37
Dupla,88537,4152,4.69%,0.65
Múltipla,20330,840,4.13%,0.58



-- condicao_metereologica --


,acidentes,fatais,taxa_letalidade,lift
condicao_metereologica,,,,
Nevoeiro/Neblina,1794,212,11.82%,1.65
Vento,328,25,7.62%,1.06
Céu Claro,135814,10124,7.45%,1.04
Nublado,32600,2269,6.96%,0.97
Chuva,20562,1267,6.16%,0.86
Sol,12059,693,5.75%,0.80
Garoa/Chuvisco,7544,411,5.45%,0.76


In [6]:
# a pior combinação e a checagem de dominância
empilhado = taxa.where(valida).stack().sort_values(ascending=False)
pior = empilhado.index[0]
print(f"Pior combinação: {pior[0]} + {pior[1]} "
      f"-> {empilhado.iloc[0]:.2%} "
      f"({int(contagem.loc[pior]):,} acidentes)")

simples_vence = 0
comparaveis = valida.loc["Simples"] & valida.loc["Dupla"]
for condicao in comparaveis[comparaveis].index:
    if taxa.loc["Simples", condicao] > taxa.loc["Dupla", condicao]:
        simples_vence += 1
print(f"\nPista Simples supera a Dupla em {simples_vence} de "
      f"{int(comparaveis.sum())} condições com volume suficiente")

assert simples_vence == int(comparaveis.sum())


Pior combinação: Simples + Nevoeiro/Neblina -> 13.95% (1,075 acidentes)

Pista Simples supera a Dupla em 7 de 7 condições com volume suficiente


**Interpretação.** O **tipo de pista domina o resultado**; o clima é secundário. Pista
**Simples** tem **9,83%** de letalidade contra **4,69%** na Dupla — **mais que o dobro** — e
taxa maior que a Dupla nas **7 de 7** condições com volume suficiente. A pior combinação é
**Simples + Nevoeiro/Neblina: 13,95%**. Volume mínimo: **50 por célula** (21 de 27 válidas).


**Detalhamento.** Na margem isolada, o resultado está **associado a** `Nevoeiro/Neblina`
(**11,82%**, lift 1,65), seguido de Vento (7,62%) e Céu Claro (7,45%) — e aí está o achado
contraintuitivo:
**chuva (6,16%) e garoa (5,45%) têm letalidade MENOR que tempo bom** (lift 0,86 e 0,76). A
leitura é de **associação**: a base não traz velocidade nem exposição, então não é possível
atribuir a diferença a um mecanismo.

Note-se que **`Ignorado` ficou fora da comparação** (2.787 acidentes, 292 fatais, taxa 10,48%,
lift 1,46) por não ser uma condição meteorológica, e que `Granizo` (6 registros) e `Neve` (3) são
volumes residuais que ficaram fora do filtro.


---
## Q9 — Top 10 BRs em **Plena Noite**: **quais** são as dez mais letais por **número de
mortos**?
Volume mínimo: 50 acidentes noturnos na rodovia.


In [7]:
noite = df[df["fase_dia"] == "Plena Noite"]
q9 = indicadores(noite, "br", minimo=50).sort_values("mortos", ascending=False)
q9["mortos_por_100_acidentes"] = q9["mortos"] / q9["acidentes"] * 100

top10 = q9.head(10)
display(fmt(top10)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]].assign(
    mortos_por_100_acidentes=top10["mortos_por_100_acidentes"].round(1)
))

print(f"Acidentes em plena noite: {len(noite):,}")
print(f"BRs presentes na noite: {noite['br'].nunique()} | "
      f"com pelo menos 50 acidentes: {len(q9)}")
print(f"Top 10 concentra {int(top10['mortos'].sum()):,} dos {int(noite['mortos'].sum()):,} "
      f"mortos noturnos ({top10['mortos'].sum() / noite['mortos'].sum():.1%})")

assert q9["mortos"].is_monotonic_decreasing
assert q9["acidentes"].min() >= 50


,acidentes,fatais,mortos,taxa_letalidade,lift,mortos_por_100_acidentes
br,,,,,,
116,11724,1033,1190,8.81%,1.23,10.2
101,12533,971,1076,7.75%,1.08,8.6
316,1524,312,342,20.47%,2.86,22.4
153,2770,281,332,10.14%,1.42,12.0
381,3160,260,285,8.23%,1.15,9.0
163,2544,220,281,8.65%,1.21,11.0
40,3240,240,265,7.41%,1.03,8.2
230,1764,228,248,12.93%,1.80,14.1
364,2161,201,229,9.30%,1.30,10.6


Acidentes em plena noite: 73,603
BRs presentes na noite: 119 | com pelo menos 50 acidentes: 83
Top 10 concentra 4,469 dos 8,535 mortos noturnos (52.4%)


In [8]:
# o ranking por mortos é o do enunciado; comparar com taxa mostra por que a escolha importa
por_taxa = q9.sort_values("taxa_letalidade", ascending=False).head(10)
print("As 10 por TAXA de letalidade (para contraste):")
display(fmt(por_taxa)[["acidentes", "mortos", "taxa_letalidade"]])

coincidem = set(top10.index) & set(por_taxa.index)
print(f"Rodovias presentes nas duas listas: {sorted(coincidem)} ({len(coincidem)} de 10)")


As 10 por TAXA de letalidade (para contraste):


,acidentes,mortos,taxa_letalidade
br,,,
402,70,24,32.86%
424,125,37,24.00%
226,349,93,22.92%
242,331,100,22.66%
222,642,153,21.96%
135,511,134,21.33%
10,542,128,21.22%
423,294,83,21.09%
316,1524,342,20.47%


Rodovias presentes nas duas listas: [316] (1 de 10)


**Interpretação.** Ordenando por **mortos**, o top 10 é dominado por **BR-116** (1.190 mortos
em 11.724 acidentes noturnos) e **BR-101** (1.076 em 12.533), que juntas respondem por **2.266
dos 8.535** mortos noturnos (26,5%); o top 10 completo concentra **52,4%** das mortes à noite.
Volume mínimo: **50 acidentes noturnos** na rodovia (83 das 119 BRs noturnas passam).


| BR | Acidentes | Mortos | Taxa | Mortos/100 ac. |
|---|---|---|---|---|
| BR-116 | 11.724 | **1.190** | 8,81% | 10,2 |
| BR-101 | 12.533 | **1.076** | 7,75% | 8,6 |
| **BR-316** | 1.524 | 342 | **20,47%** | **22,4** |
| BR-153 | 2.770 | 332 | 10,14% | 12,0 |
| BR-381 | 3.160 | 285 | 8,23% | 9,0 |
| BR-163 | 2.544 | 281 | 8,65% | 11,0 |
| BR-040 | 3.240 | 265 | 7,41% | 8,2 |
| BR-230 | 1.764 | 248 | 12,93% | 14,1 |
| BR-364 | 2.161 | 229 | 9,30% | 10,6 |
| BR-277 | 2.269 | 221 | 8,64% | 9,7 |


**A escolha do critério muda a resposta.** As duas listas quase não se cruzam: só a
**BR-316** aparece em ambas. BR-116 e BR-101 ocupam as duas primeiras posições por
mortos e também são as rodovias de **maior volume noturno** da base — o ranking diz onde
morre mais gente em termos absolutos, não onde o risco por acidente é maior. A BR-316 tem
**menos de um sétimo** dos acidentes da BR-116, porém taxa de **20,47%** (lift 2,86) e
**22,4 mortos por 100 acidentes** — mais que o dobro da BR-116 (10,2).

Portanto: para **priorizar socorro e fiscalização por volume**, o ranking por mortos é o
correto e a BR-116 vem primeiro. Para **entender onde cada acidente mata mais**, a BR-316
é o caso a investigar. As duas leituras respondem perguntas diferentes.


---
## Q10 — Fim de Ano, Carnaval e dias normais:
a taxa de letalidade sobe nos feriados? Compare também a média diária de acidentes.


In [9]:
q10 = indicadores(df, "data_comemorativa", minimo=1)
q10 = q10.join(
    df.groupby("data_comemorativa")["data_inversa"].nunique().rename("dias")
)
q10["acidentes_por_dia"] = q10["acidentes"] / q10["dias"]
q10["mortos_por_dia"] = q10["mortos"] / q10["dias"]
q10 = q10.sort_values("taxa_letalidade", ascending=False)

tabela_q8 = fmt(q10)[["dias", "acidentes", "fatais", "mortos",
                      "taxa_letalidade", "lift"]].assign(
    acidentes_por_dia=q10["acidentes_por_dia"].round(1),
    mortos_por_dia=q10["mortos_por_dia"].round(2),
)
display(tabela_q8)
print("O volume de cada período já é o total observado: não há filtro de mínimo.")


,dias,acidentes,fatais,mortos,taxa_letalidade,lift,acidentes_por_dia,mortos_por_dia
data_comemorativa,,,,,,,,
Fim de Ano,42,8863,656,849,7.40%,1.03,211.0,20.21
Normal,1039,201737,14445,16779,7.16%,1.00,194.2,16.15
Carnaval,15,2897,192,205,6.63%,0.93,193.1,13.67


O volume de cada período já é o total observado: não há filtro de mínimo.


In [10]:
# comparação explícita contra a referência "Normal"
normal = q10.loc["Normal"]
print("Comparação com dias normais:\n")
for periodo in ("Fim de Ano", "Carnaval"):
    linha = q10.loc[periodo]
    dif_taxa = (linha["taxa_letalidade"] - normal["taxa_letalidade"]) * 100
    dif_ac = (linha["acidentes_por_dia"] / normal["acidentes_por_dia"] - 1) * 100
    print(f"{periodo:<12} taxa {linha['taxa_letalidade']:>6.2%} "
          f"({dif_taxa:+.2f} p.p. vs normal) | "
          f"acidentes/dia {linha['acidentes_por_dia']:>5.1f} "
          f"({dif_ac:+.1f}% vs normal) | "
          f"mortos/dia {linha['mortos_por_dia']:.2f}")

assert q10.loc["Fim de Ano", "acidentes_por_dia"] > normal["acidentes_por_dia"]


Comparação com dias normais:

Fim de Ano   taxa  7.40% (+0.24 p.p. vs normal) | acidentes/dia 211.0 (+8.7% vs normal) | mortos/dia 20.21
Carnaval     taxa  6.63% (-0.53 p.p. vs normal) | acidentes/dia 193.1 (-0.5% vs normal) | mortos/dia 13.67


**Interpretação.** Nos três períodos a letalidade é **praticamente a mesma**: Fim de Ano
**7,40%** (lift 1,03), Normal **7,16%** (referência) e Carnaval **6,63%** (lift 0,93). A
amplitude é de apenas **0,77 p.p.** — não há evidência de que o feriado por si só torne as
viagens mais letais. O que muda de forma clara é o **volume diário** (tabela abaixo).


| Período | Dias | Acidentes/dia | Mortos/dia | Taxa |
|---|---|---|---|---|
| Fim de Ano | 42 | **211,0** (+8,7%) | **20,21** (+25,2%) | 7,40% |
| Normal | 1.039 | 194,2 | 16,15 | 7,16% |
| Carnaval | 15 | 193,1 (−0,5%) | 13,67 (−15,4%) | 6,63% |


O Fim de Ano concentra **mais acidentes por dia** e, sobretudo, **mais mortos por dia**;
a taxa quase não muda porque o número de acidentes cresce junto. O Carnaval, por outro
lado, tem acidentes por dia praticamente iguais aos de um dia normal (**−0,53%**), mas
**15,37% menos mortos por dia** (13,67 contra 16,15) e a **menor** letalidade dos três.

Duas ressalvas necessárias: o Carnaval tem só **15 dias** em três anos, o que deixa a taxa
sensível a poucos casos; e o Fim de Ano inclui 20/12 a 02/01, intervalo em que a exposição
viária cresce por motivos alheios ao feriado.


---
## Fechamento do Nível 3

| Questão | Achado |
|---|---|
| Q8 | Pista Simples (9,83%) mais que dobra a Dupla (4,69%) em todas as 7 condições; pior célula é Simples + Nevoeiro (13,95%); chuva tem lift **abaixo** de 1 |
| Q9 | Por mortos: BR-116 (1.190) e BR-101 (1.076) lideram, 52,4% das mortes noturnas no top 10; só a BR-316 aparece também no ranking por taxa |
| Q10 | Letalidade quase idêntica nos três períodos (6,63% a 7,40%); Fim de Ano tem +8,7% de acidentes/dia e +25,2% de mortos/dia; Carnaval tem acidentes/dia iguais ao normal (−0,5%) mas 15,4% menos mortos/dia |
